In [3]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key="pcsk_4oFMz4_35gaFqZKaWMkbxLvco9CCm7wx9b9XcrRQLbsEFgRyhBdyUmmDXxeTYbrjttsAsz")

index_name = "vec-0601"
namespace = ""  # 沒有 namespace 就留空；有的話改成你的 namespace

index = pc.Index(index_name)

# 1. 看總筆數
stats = index.describe_index_stats()

print("=== Index Stats ===")
print(stats)
print("總筆數:", stats["total_vector_count"])

print("\n=== Namespace 筆數 ===")
for ns, info in stats.get("namespaces", {}).items():
    print(f"{ns}: {info['vector_count']}")

# 2. 列出所有 _id
print("\n=== Vector IDs ===")
all_ids = []

for ids in index.list(namespace=namespace):
    all_ids.extend(ids)

print("ID 數量:", len(all_ids))
print(all_ids[:20])  # 先看前 20 個

=== Index Stats ===
DescribeIndexStatsResponse(dimension=1024, total_vector_count=111, metric='cosine', namespaces=1)
總筆數: 111

=== Namespace 筆數 ===
__default__: 111

=== Vector IDs ===
ID 數量: 111
[ListItem(id='07oMbIid9WVaRcnGPX4Q'), ListItem(id='18wpK9VDO5RuYemvHIJk'), ListItem(id='1dOSN6uJcSl5ih4vfEF6'), ListItem(id='1jkcjq8BVFeReTLBQr9u'), ListItem(id='30SsgkbeUbcC8N9JMtgx'), ListItem(id='3JsE5ORznE0fYqQpUthM'), ListItem(id='3Y855cSAtGRgl1yxUwqK'), ListItem(id='3bJP8QjDr7yA4zVwejdl'), ListItem(id='4BNZ0DSvM0cdYYLdIsxI'), ListItem(id='4ZFPriLeSWPo8rJAOroO'), ListItem(id='4njJAGdtsQwe7gsbxO5I'), ListItem(id='53leeil0YLV9EhKRj97k'), ListItem(id='5ssJYLBsvPGtx2Ht9oKZ'), ListItem(id='72y30EHstjAvpYnAM2wt'), ListItem(id='7OFWhLA4a6xcDSAduKHd'), ListItem(id='7XYUzD6VC4VuOHEC2BcI'), ListItem(id='8lArnQKucq9QLpe2ofBN'), ListItem(id='97V38cFljrz1B2pk0SYC'), ListItem(id='AMo1HCVb9uPPMimfMjyX'), ListItem(id='AYzcwj19QvZFIBn8SHKi')]


In [1]:
import json

with open(
    "./eval_ragas_testset/docs/docs_cache_2026-06-03.json", "r", encoding="utf-8"
) as f:
    data = json.load(f)

print(len(data))

106


In [11]:
import json
import os
from pinecone import Pinecone

# 1. 讀 JSON
with open(
    "./eval_ragas_testset/docs/docs_cache_2026-06-03.json", "r", encoding="utf-8"
) as f:
    data = json.load(f)

json_ids = {str(item["id"]) for item in data}

print("JSON id 數量:", len(json_ids))


# 2. 讀 Pinecone ids
# pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
# index = pc.Index("你的-index-name")

namespace = ""

pinecone_ids = set()

for ids in index.list(namespace=namespace):
    for pid in ids:
        pinecone_ids.add(pid.id)  # 重點是這裡，不要用 str(pid)

print("Pinecone id 數量:", len(pinecone_ids))


# 3. 比對
json_has_pinecone_not = json_ids - pinecone_ids
pinecone_has_json_not = pinecone_ids - json_ids

print("\n=== JSON 有，但 Pinecone 沒有 ===")
print("數量:", len(json_has_pinecone_not))
for id_ in sorted(json_has_pinecone_not):
    print(id_)

print("\n=== Pinecone 有，但 JSON 沒有 ===")
print("數量:", len(pinecone_has_json_not))
for id_ in sorted(pinecone_has_json_not):
    print(id_)

JSON id 數量: 106
Pinecone id 數量: 106

=== JSON 有，但 Pinecone 沒有 ===
數量: 0

=== Pinecone 有，但 JSON 沒有 ===
數量: 0


In [13]:
import json
from datetime import datetime
import firebase_admin
from firebase_admin import credentials, firestore

# 初始化 Firebase
cred = credentials.Certificate("serviceAccountKey.json")

# 避免 ipynb 重複執行報錯
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()

teaching_material_posts = []
non_teaching_material_posts = []

docs = db.collection("posts").stream()

for doc in docs:
    data = doc.to_dict()
    data["id"] = doc.id

    # Firestore Timestamp / datetime 轉成 JSON 可存的格式
    for key, value in data.items():
        if isinstance(value, datetime):
            data[key] = value.isoformat()

    # 只有明確 isTeachingMaterial == True 才放這邊
    if data.get("isTeachingMaterial") is True:
        teaching_material_posts.append(data)
    else:
        non_teaching_material_posts.append(data)

# 輸出 JSON
with open("posts_isTeachingMaterial_true.json", "w", encoding="utf-8") as f:
    json.dump(teaching_material_posts, f, ensure_ascii=False, indent=2)

with open("posts_isTeachingMaterial_false_or_missing.json", "w", encoding="utf-8") as f:
    json.dump(non_teaching_material_posts, f, ensure_ascii=False, indent=2)

print("isTeachingMaterial == true:", len(teaching_material_posts))
print("不是 true 或沒有欄位:", len(non_teaching_material_posts))

isTeachingMaterial == true: 22
不是 true 或沒有欄位: 84


In [15]:
import json
import os
from pinecone import Pinecone

# pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
# index = pc.Index("你的-index-name")

# 如果 describe_index_stats() 顯示 namespace 是 ''，就用空字串
namespace = ""

with open("posts_isTeachingMaterial_true.json", "r", encoding="utf-8") as f:
    teaching_posts = json.load(f)

with open("posts_isTeachingMaterial_false_or_missing.json", "r", encoding="utf-8") as f:
    sharing_posts = json.load(f)

teaching_ids = [str(item["id"]) for item in teaching_posts]
sharing_ids = [str(item["id"]) for item in sharing_posts]

# 教學教材
for id_ in teaching_ids:
    index.update(id=id_, set_metadata={"sourceType": "teaching_material"})

# 他人分享
for id_ in sharing_ids:
    index.update(id=id_, set_metadata={"sourceType": "peer_sharing"})

print("更新 教學教材:", len(teaching_ids))
print("更新 他人分享:", len(sharing_ids))

更新 教學教材: 22
更新 他人分享: 84
